# 04. デモ

学習済みモデルを使って、医療テキストから**疾患名・薬剤名**を抽出します。

## エンティティ
- **疾患（Disease）**: 🟠 薄いオレンジ
- **薬剤（Chemical）**: 🔵 薄い青

## 1. ライブラリインポート

In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
from IPython.display import HTML, display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA利用可: {torch.cuda.is_available()}")

PyTorch: 2.5.1+cu121
CUDA利用可: True


## 2. モデルとトークナイザーのロード

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "../results/biobert-ner-bc5cdr"

# トークナイザーとモデルをロード
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# ラベル定義を確認
label_list = list(model.config.id2label.values())

print(f"モデル: {model_path}")
print(f"デバイス: {device}")
print(f"\n=== ラベル定義 ===")
for i, label in enumerate(label_list):
    icon = ''
    if 'Chemical' in label:
        icon = '🔵'
    elif 'Disease' in label:
        icon = '🟠'
    print(f"{i}: {label} {icon}")

モデル: ../results/biobert-ner-bc5cdr
デバイス: cuda

=== ラベル定義 ===
0: O 
1: B-Chemical 🔵
2: I-Chemical 🔵
3: B-Disease 🟠
4: I-Disease 🟠


## 3. NERパイプラインの初期化

In [3]:
# パイプライン作成
# aggregation_strategy="first" は WordPiece サブワードを正しく結合
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",
    device=0 if device == "cuda" else -1
)

print("パイプライン初期化完了")
print(f"aggregation_strategy: first (WordPiece サブワード結合用)")

パイプライン初期化完了
aggregation_strategy: first (WordPiece サブワード結合用)


## 4. 単一テキストでテスト

In [4]:
# テスト
test_text = "The patient has diabetes mellitus and takes insulin for treatment."
entities = ner_pipeline(test_text)

print(f"テキスト: {test_text}\n")
print("抽出結果:")
if entities:
    for ent in entities:
        icon = '🟠' if 'Disease' in ent['entity_group'] else '🔵'
        print(f"  - {ent['entity_group']}: {ent['word']} {icon} (スコア: {ent['score']:.4f})")
else:
    print("  (エンティティが見つかりませんでした)")

テキスト: The patient has diabetes mellitus and takes insulin for treatment.

抽出結果:
  - Disease: diabetes mellitus 🟠 (スコア: 0.8559)
  - Chemical: insulin 🔵 (スコア: 0.6995)


## 5. 複数サンプルでテスト

In [5]:
# 複数のサンプル
sample_texts = [
    "Patient has diabetes mellitus and takes insulin for treatment.",
    "The patient was diagnosed with hypertension and heart disease.",
    "Treatment includes aspirin for prevention of myocardial infarction.",
    "Clinical trials show efficacy of metformin in patients with pneumonia.",
    "The patient suffers from arthritis and uses ibuprofen for pain relief."
]

for i, text in enumerate(sample_texts, 1):
    entities = ner_pipeline(text)
    print(f"\n[Sample {i}]")
    print(f"Text: {text}")
    print("Entities:")
    if entities:
        for ent in entities:
            icon = '🟠' if 'Disease' in ent['entity_group'] else '🔵'
            print(f"  - {ent['entity_group']}: {ent['word']} {icon} (スコア: {ent['score']:.4f})")
    else:
        print("  (なし)")


[Sample 1]
Text: Patient has diabetes mellitus and takes insulin for treatment.
Entities:
  - Disease: diabetes mellitus 🟠 (スコア: 0.8584)
  - Chemical: insulin 🔵 (スコア: 0.7099)

[Sample 2]
Text: The patient was diagnosed with hypertension and heart disease.
Entities:
  - Disease: hypertension 🟠 (スコア: 0.9600)
  - Disease: heart disease 🟠 (スコア: 0.8982)

[Sample 3]
Text: Treatment includes aspirin for prevention of myocardial infarction.
Entities:
  - Chemical: aspirin 🔵 (スコア: 0.9902)
  - Disease: myocardial infarction 🟠 (スコア: 0.9515)

[Sample 4]
Text: Clinical trials show efficacy of metformin in patients with pneumonia.
Entities:
  - Chemical: metformin 🔵 (スコア: 0.9800)
  - Disease: pneumonia 🟠 (スコア: 0.8013)

[Sample 5]
Text: The patient suffers from arthritis and uses ibuprofen for pain relief.
Entities:
  - Disease: arthritis 🟠 (スコア: 0.8877)
  - Chemical: ibuprofen 🔵 (スコア: 0.9871)
  - Disease: pain 🟠 (スコア: 0.8324)


## 6. ハイライト表示

In [6]:
def highlight_entities(text, entities):
    """エンティティをハイライト表示"""
    # エンティティを位置でソート（後ろから処理）
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
    
    highlighted = text
    for ent in sorted_entities:
        word = ent['word']
        start = ent['start']
        end = ent['end']
        entity_type = ent['entity_group']
        
        if 'Disease' in entity_type:
            color = '#ffe6b3'  # 薄いオレンジ
            icon = '🟠'
        else:
            color = '#b3d9ff'  # 薄い青
            icon = '🔵'
        
        # ハイライト追加
        highlighted = highlighted[:end] + f"</span>" + highlighted[end:]
        highlighted = highlighted[:start] + f"<span style='background-color:{color};padding:2px 4px;border-radius:3px;'>{word}" + highlighted[start+len(word):]
    
    return highlighted

# サンプルをハイライト表示
for i, text in enumerate(sample_texts[:3], 1):
    entities = ner_pipeline(text)
    highlighted = highlight_entities(text, entities)
    
    print(f"\n[Sample {i}]")
    display(HTML(f"<p style='font-size:14px;'>{highlighted}</p>"))


[Sample 1]



[Sample 2]



[Sample 3]


## 7. 集計

In [7]:
# すべてのサンプルで集計
all_entities = []
for text in sample_texts:
    entities = ner_pipeline(text)
    all_entities.extend(entities)

diseases = [e['word'] for e in all_entities if 'Disease' in e['entity_group']]
chemicals = [e['word'] for e in all_entities if 'Chemical' in e['entity_group']]

print("=== 集計結果 ===")
print(f"疾患（Disease 🟠）: {len(diseases)}")
print(f"薬剤（Chemical 🔵）: {len(chemicals)}")
print(f"\n疾患リスト: {set(diseases) if diseases else '(なし)'}")
print(f"薬剤リスト: {set(chemicals) if chemicals else '(なし)'}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== 集計結果 ===
疾患（Disease 🟠）: 7
薬剤（Chemical 🔵）: 4

疾患リスト: {'myocardial infarction', 'hypertension', 'heart disease', 'pain', 'diabetes mellitus', 'pneumonia', 'arthritis'}
薬剤リスト: {'aspirin', 'metformin', 'ibuprofen', 'insulin'}


---

デモ完了